In [6]:
import pandas as pd
import numpy as np

In [1]:
from preprocessing import load_and_prepare, remove_outliers
DATA = "../data"

In [2]:
data = load_and_prepare(folder=DATA)

gaps: 0.467->0.811 and 1.343->2.141 | cutoffs 0.639, 1.742 | dropped 532 of 38330
gaps: 0.466->0.807 and 1.338->2.140 | cutoffs 0.637, 1.739 | dropped 677 of 48000
split cutoff: 2025-08-31
train       (37798, 13)
test        (9670, 13)
full        (47323, 13)
validation  (12000, 13)
chart       (31, 13)


In [3]:
X_train, y_train = data["X_train"], data["y_train"]
X_test, y_test = data["X_test"], data["y_test"]
X_full, y_full = data["X_full"], data["y_full"]
X_val, X_chart = data["X_val"], data["X_chart"]
train, test = data["train"], data["test"]

In [15]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 37798 entries, TR-000001 to TR-038234
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   pickup_lat         37798 non-null  float64
 1   pickup_lon         37798 non-null  float64
 2   delivery_lat       37798 non-null  float64
 3   delivery_lon       37798 non-null  float64
 4   distance           37798 non-null  float64
 5   weight             37798 non-null  float64
 6   market_index       37798 non-null  float64
 7   quote_signal       37798 non-null  float64
 8   day_of_week        37798 non-null  int32  
 9   month              37798 non-null  int32  
 10  equipment_Dry Van  37798 non-null  int64  
 11  equipment_Flatbed  37798 non-null  int64  
 12  equipment_Reefer   37798 non-null  int64  
dtypes: float64(8), int32(2), int64(3)
memory usage: 4.8+ MB


In [16]:
X_val.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12000 entries, TE-000001 to TE-012000
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   pickup_lat         12000 non-null  float64
 1   pickup_lon         12000 non-null  float64
 2   delivery_lat       12000 non-null  float64
 3   delivery_lon       12000 non-null  float64
 4   distance           12000 non-null  float64
 5   weight             12000 non-null  float64
 6   market_index       12000 non-null  float64
 7   quote_signal       12000 non-null  float64
 8   day_of_week        12000 non-null  int32  
 9   month              12000 non-null  int32  
 10  equipment_Dry Van  12000 non-null  int64  
 11  equipment_Flatbed  12000 non-null  int64  
 12  equipment_Reefer   12000 non-null  int64  
dtypes: float64(8), int32(2), int64(3)
memory usage: 1.4+ MB


In [4]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

clean_idx = remove_outliers(test, verbose=False).index

def evaluate(name, pred):
    pred = pd.Series(pred, index=X_test.index)
    mae_all = mean_absolute_error(y_test, pred)
    mae_clean = mean_absolute_error(y_test[clean_idx], pred[clean_idx])
    rmse_clean = root_mean_squared_error(y_test[clean_idx], pred[clean_idx])
    print(f"{name:18s} | MAE all ${mae_all:7.1f} | MAE clean ${mae_clean:6.1f} | RMSE clean ${rmse_clean:6.1f}")

In [7]:
BANDS = [0, 300, 800, 1500, 4000]
rpm = train["posted_rate"] / train["distance"]
typical = rpm.groupby([train["equipment"], pd.cut(train["distance"], BANDS)], observed=True).median()

keys = pd.MultiIndex.from_arrays([test["equipment"], pd.cut(test["distance"], BANDS)])
evaluate("baseline", typical.reindex(keys).values * test["distance"].values)

baseline           | MAE all $  147.1 | MAE clean $  92.0 | RMSE clean $ 126.3


In [17]:
import time
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from scipy.stats import randint, uniform, loguniform
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

cv = TimeSeriesSplit(n_splits=3)

searches = {
    "hist_gb": (
        HistGradientBoostingRegressor(early_stopping=False, random_state=42),
        {
            "loss": ["squared_error", "absolute_error"],
            "learning_rate": loguniform(0.02, 0.2),
            "max_iter": randint(200, 1000),
            "max_leaf_nodes": randint(15, 127),
            "min_samples_leaf": randint(10, 100),
            "l2_regularization": loguniform(1e-3, 10),
        },
        15,
    ),
    "xgboost": (
        XGBRegressor(random_state=42, n_jobs=-1),
        {
            "objective": ["reg:squarederror", "reg:absoluteerror"],
            "learning_rate": loguniform(0.02, 0.2),
            "n_estimators": randint(200, 1000),
            "max_depth": randint(4, 10),
            "min_child_weight": randint(1, 20),
            "subsample": uniform(0.6, 0.4),          # between 0.6 and 1.0
            "colsample_bytree": uniform(0.6, 0.4),
        },
        15,
    ),
    "lightgbm": (
        LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1, subsample_freq=1),
        {
            "objective": ["l2", "l1"],
            "learning_rate": loguniform(0.02, 0.2),
            "n_estimators": randint(200, 1000),
            "num_leaves": randint(15, 127),
            "min_child_samples": randint(10, 100),
            "subsample": uniform(0.6, 0.4),
            "colsample_bytree": uniform(0.6, 0.4),
        },
        15,
    ),
    "random_forest": (
        RandomForestRegressor(random_state=42, n_jobs=-1),
        {
            "n_estimators": randint(100, 300),
            "max_depth": [None, 12, 20],
            "min_samples_leaf": randint(1, 20),
            "max_features": uniform(0.4, 0.6),       # between 0.4 and 1.0
        },
        6,                                           # fewer tries: much slower to train
    ),
}

In [18]:
results = {}

for name, (model, space, n_iter) in searches.items():
    start = time.time()
    search = RandomizedSearchCV(
        model, space, n_iter=n_iter, cv=cv,
        scoring="neg_mean_absolute_error", random_state=42,
    )
    search.fit(X_train, y_train)

    results[name] = search
    print(f"{name:14s} CV MAE ${-search.best_score_:.1f} | {time.time() - start:.0f}s")
    print("   best:", search.best_params_)

hist_gb        CV MAE $86.4 | 116s
   best: {'l2_regularization': np.float64(0.0010672476836323724), 'learning_rate': np.float64(0.02109076927540637), 'loss': 'squared_error', 'max_iter': 258, 'max_leaf_nodes': 56, 'min_samples_leaf': 69}
xgboost        CV MAE $78.6 | 35s
   best: {'colsample_bytree': np.float64(0.836965827544817), 'learning_rate': np.float64(0.022257706349811456), 'max_depth': 6, 'min_child_weight': 7, 'n_estimators': 220, 'objective': 'reg:squarederror', 'subsample': np.float64(0.6260206371941118)}
lightgbm       CV MAE $73.6 | 52s
   best: {'colsample_bytree': np.float64(0.7760609974958406), 'learning_rate': np.float64(0.0264891626801987), 'min_child_samples': 17, 'n_estimators': 234, 'num_leaves': 92, 'objective': 'l2', 'subsample': np.float64(0.7035119926400067)}
random_forest  CV MAE $81.7 | 18s
   best: {'max_depth': 20, 'max_features': np.float64(0.9197056874649611), 'min_samples_leaf': 4, 'n_estimators': 203}


In [19]:
for name, s in results.items():
    evaluate(name, s.best_estimator_.predict(X_test))

hist_gb            | MAE all $  111.8 | MAE clean $  56.3 | RMSE clean $  85.0
xgboost            | MAE all $  115.8 | MAE clean $  60.4 | RMSE clean $  90.7
lightgbm           | MAE all $  111.8 | MAE clean $  56.4 | RMSE clean $  83.7
random_forest      | MAE all $  111.6 | MAE clean $  56.0 | RMSE clean $  84.4


# Stage 2 with the full data

In [20]:
from sklearn.base import clone

final_model = clone(results["lightgbm"].best_estimator_)
final_model.fit(X_full, y_full)

,boosting_type,'gbdt'
,num_leaves,92
,max_depth,-1
,learning_rate,np.float64(0.0264891626801987)
,n_estimators,234
,subsample_for_bin,200000
,objective,'l2'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,17
